## SQL

![dbschema](assets/db_schema.png)

### 0. Initialization

Import the required libraries and establish a connection with postgres database in your docker compose stack.

In [16]:
import psycopg2
import pandas as pd

conn = psycopg2.connect(
    dbname="postgres",
    user="postgres",
    password="ulfriids",
    host="localhost",
    port="5432"
)

# list all tables
query = """
SELECT table_name
FROM information_schema.tables
WHERE table_schema = 'public'
"""
df = pd.read_sql_query(query, conn)
df

/tmp/ipykernel_13256/1670175545.py:18: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(query, conn)


,table_name
0,loan_applications
1,applicants


### How many loan applications were approved/rejected?

In [17]:
query = """
    SELECT COUNT(*), la.loan_approved
    FROM loan_applications la
    GROUP BY la.loan_approved
    ORDER BY la.loan_approved
"""

df = pd.read_sql_query(query, conn)
df

/tmp/ipykernel_13256/3716328935.py:8: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(query, conn)


,count,loan_approved
0,35000,0
1,10000,1


### 1. Write a query to join both the tables loan_applications and applicants.

In [18]:
query = """
    SELECT *
    FROM applicants a
    JOIN loan_applications la ON la.applicant_id = a.applicant_id
    """

df = pd.read_sql_query(query, conn)
df

/tmp/ipykernel_13256/4067082839.py:7: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(query, conn)


,applicant_id,age,gender,education_level,income,years_work_experience,owns_home,credit_history_length_years,credit_score,previous_loan_defaults,applicant_id,loan_amount,loan_interest_rate,loan_percent_income,loan_approved
0,1,22,female,Master,71948,0,0,3,561,0,1,35000,16.02,0.49,1
1,2,21,female,High School,12282,0,1,2,504,1,2,1000,11.14,0.08,0
2,3,25,female,High School,12438,3,1,3,635,0,3,5500,12.87,0.44,1
3,4,23,female,Bachelor,79753,0,0,2,675,0,4,35000,15.23,0.44,1
4,5,24,male,Master,66135,1,0,4,586,0,5,35000,14.27,0.53,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
44995,44996,27,male,Associate,47971,6,0,3,645,0,44996,15000,15.66,0.31,1
44996,44997,37,female,Associate,65800,17,0,11,621,0,44997,9000,14.07,0.14,1
44997,44998,33,male,Associate,56942,7,0,10,668,0,44998,2771,10.02,0.05,1
44998,44999,29,male,Bachelor,33164,4,0,6,604,0,44999,12000,13.23,0.36,1


### 2. Write an sql query to reproduce the result of the following table.

![alt text](assets/query_expected_result.jpg)

In [19]:
query = """
    SELECT a.education_level, AVG(la.loan_amount) AS avg_loan_amount
    FROM loan_applications la
    JOIN applicants a ON la.applicant_id = a.applicant_id
    WHERE a.age BETWEEN 25 AND 45
    GROUP BY a.education_level
    ORDER BY avg_loan_amount DESC
    """

df = pd.read_sql_query(query, conn)
df

/tmp/ipykernel_13256/634528411.py:10: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(query, conn)


,education_level,avg_loan_amount
0,Associate,9979.653951
1,Doctorate,9964.831325
2,High School,9894.870726
3,Master,9888.460697
4,Bachelor,9808.688614


### 3. Predicting loan application approval/rejection

Your goal is to predict if a bank should approve a loan application. 

First prepare the dataset and save it to csv

FEATURE_COLUMNS: `loan_amount, loan_interest_rate, loan_percent_income, age, income, years_work_experience, owns_home, credit_history_length_years, credit_score, previous_loan_defaults`

TARGET_COLUMN: `loan_approved`

Use the cell below to run both queries to obtain a feature dataframe X and a target dataframe y

You shall start by establishing a baseline: a dummy classifier that always predicts the majority class. Evaluate the dummy classifier with the `classification_report()` provided by scikit. Finally, train a RandomForestClassifier and compare it to the baseline with the same `classification_report()`

Useful links: 
- [sklearn.dummy.DummyClassifier](https://scikit-learn.org/stable/modules/generated/sklearn.dummy.DummyClassifier.html) and [sklearn dummy estimators user guide](https://scikit-learn.org/stable/modules/model_evaluation.html#dummy-estimators).
- [sklearn.model_selection.train_test_split](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html)
- [sklearn.metrics.classification_report](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.classification_report.html)
- [sklearn.ensemble.RandomForestClassifier](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestClassifier.html)